# Quora Question Pairs - Duplicate Detection (DeBERTa-v3-base, 3 epochs, GPU T4 x2)
Dataset: `questions.csv` (`id, qid1, qid2, question1, question2, is_duplicate`).
Split: **80% train / 10% validation / 10% test** (stratified).
- Validation: picks the best epoch + F1-optimal threshold
- Test: final untouched F1

**Settings:** Accelerator = GPU T4 x2, Internet = ON.
Every cell shows a `tqdm` progress bar. The red "LOAD REPORT" box when the model loads is a normal warning, not an error.

**Kaggle resource budget (this config):** ~4-6 GB per GPU (of 15 GB), ~5 GB RAM (of 30 GB), ~2 GB disk (of 57 GB),
roughly 2-3 hours total (session limit is 12 h).

**v2:** adds a preprocessing + train-only augmentation cell (toggle each step in the `PREP` dict).

**v3:** adds a Stage-2 stacking cell after the test evaluation. It prints DeBERTa alone vs. DeBERTa + features on the SAME test rows, plus a precision/recall threshold table.
Note: the question-graph features (`deg1`, `deg2`, `common_nbrs`) exploit how this dataset was sampled, not language understanding. Compare the 'text features' and 'graph features' lines to see how much each adds.


In [1]:
import os, glob, time, random, gc
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, log_loss, classification_report, accuracy_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

CFG = dict(
    model_name="microsoft/deberta-v3-base",
    max_len=96,        # dynamic padding -> only long pairs pay the extra cost
    epochs=3,
    lr=2e-5,
    train_bs=64,       # fits comfortably in 15 GB per T4 with fp16 (split across the 2 GPUs)
    eval_bs=256,
    warmup=0.06,
    wd=0.01,
    seed=42,
    tta=True,          # average predictions over both question orders
)

def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def gpu_mem():
    return " | ".join(f"GPU{i}: {torch.cuda.memory_allocated(i)/2**30:.1f}/{torch.cuda.get_device_properties(i).total_memory/2**30:.0f} GiB"
                      for i in range(torch.cuda.device_count()))

with tqdm(total=2, desc="Setup") as bar:
    seed_everything(CFG["seed"]); bar.update(1)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu"); bar.update(1)
print("Device:", device, "| GPUs:", torch.cuda.device_count())

Setup:   0%|          | 0/2 [00:00<?, ?it/s]

Device: cuda | GPUs: 2


In [2]:
# ---------------- Load data + 80/10/10 split ----------------
with tqdm(total=4, desc="Data loading") as bar:
    path = sorted(glob.glob("/kaggle/input/**/questions.csv", recursive=True))[0]; bar.update(1)
    df = pd.read_csv(path); bar.update(1)
    df = df.dropna(subset=["question1", "question2"]).reset_index(drop=True)      # drops the 2 rows with missing text
    df["question1"] = df["question1"].astype(str); df["question2"] = df["question2"].astype(str); bar.update(1)
    tr_df, tmp_df = train_test_split(df, test_size=0.2, stratify=df.is_duplicate, random_state=CFG["seed"])
    va_df, te_df  = train_test_split(tmp_df, test_size=0.5, stratify=tmp_df.is_duplicate, random_state=CFG["seed"])
    tr_df, va_df, te_df = [d.reset_index(drop=True) for d in (tr_df, va_df, te_df)]; bar.update(1)

print("Loaded:", path)
print(df.shape, "| duplicate rate:", round(df.is_duplicate.mean(), 3))
print("train/val/test:", len(tr_df), len(va_df), len(te_df))

Data loading:   0%|          | 0/4 [00:00<?, ?it/s]

Loaded: /kaggle/input/datasets/quora/question-pairs-dataset/questions.csv
(404348, 6) | duplicate rate: 0.369
train/val/test: 323478 40435 40435


In [3]:
# ---------------- Extra preprocessing + train-only augmentation ----------------
import html, re, unicodedata
from collections import defaultdict

PREP = dict(
    clean_text=True,        # light, transformer-safe cleaning (keeps case, punctuation, numbers)
    drop_conflicts=True,    # drop train pairs that appear twice with different labels
    augment=True,           # transitive-closure augmentation (TRAIN split only, so no leakage)
    max_aug_pos=20000,      # cap on extra duplicate pairs
    max_aug_neg=20000,      # cap on extra non-duplicate pairs
)

def clean(t):
    t = html.unescape(str(t))
    t = unicodedata.normalize("NFKC", t)
    t = t.replace("\u2019", "'").replace("\u2018", "'").replace("\u201c", '"').replace("\u201d", '"')
    t = re.sub(r"\[/?math\]", " ", t)           # Quora math tags -> keep the content
    t = re.sub(r"\s+", " ", t).strip()
    return t

def transitive_augment(d, max_pos, max_neg, seed=42):
    """If A~B and B~C then A~C. If A~B and B!~C then A!~C. Uses TRAIN pairs only."""
    rng = random.Random(seed)
    qtext = {}
    for a, b, ta, tb in zip(d.qid1, d.qid2, d.question1, d.question2):
        qtext[a], qtext[b] = ta, tb
    parent = {q: q for q in qtext}
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]; x = parent[x]
        return x
    for a, b, y in zip(d.qid1, d.qid2, d.is_duplicate):
        if y == 1: parent[find(a)] = find(b)
    comp = defaultdict(list)
    for q in qtext: comp[find(q)].append(q)
    existing = {frozenset((a, b)) for a, b in zip(d.qid1, d.qid2)}

    pos = []
    for members in comp.values():
        if len(members) < 3: continue
        pairs = [(members[i], members[j]) for i in range(len(members)) for j in range(i + 1, len(members))] \
                if len(members) <= 8 else [tuple(rng.sample(members, 2)) for _ in range(20)]
        pos += [p for p in pairs if frozenset(p) not in existing]
    neg = []
    for a, b, y in zip(d.qid1, d.qid2, d.is_duplicate):
        if y == 1: continue
        ra, rb = find(a), find(b)
        if ra == rb: continue                      # contradictory labels -> skip
        A, B = comp[ra], comp[rb]
        if len(A) == 1 and len(B) == 1: continue
        for _ in range(3):
            x, z = rng.choice(A), rng.choice(B)
            if frozenset((x, z)) not in existing and (x, z) != (a, b):
                neg.append((x, z))
    pos = list(set(pos)); neg = list(set(neg))
    rng.shuffle(pos); rng.shuffle(neg)
    pos, neg = pos[:max_pos], neg[:max_neg]
    rows = [(qtext[x], qtext[z], 1) for x, z in pos] + [(qtext[x], qtext[z], 0) for x, z in neg]
    return pd.DataFrame(rows, columns=["question1", "question2", "is_duplicate"]), len(pos), len(neg)

with tqdm(total=3, desc="Preprocessing") as bar:
    if PREP["clean_text"]:
        for d in (tr_df, va_df, te_df):
            d["question1"] = d.question1.map(clean); d["question2"] = d.question2.map(clean)
    bar.update(1)

    n0 = len(tr_df)
    keep = (tr_df.question1.str.len() > 0) & (tr_df.question2.str.len() > 0)
    tr_df = tr_df[keep]
    if PREP["drop_conflicts"]:
        key = tr_df.apply(lambda r: " || ".join(sorted((r.question1.lower(), r.question2.lower()))), axis=1)
        nlab = tr_df.groupby(key.values).is_duplicate.transform("nunique")
        tr_df = tr_df[nlab.values == 1].drop_duplicates(subset=["qid1", "qid2"])
    tr_df = tr_df.reset_index(drop=True)
    print(f"Cleaned train: {n0} -> {len(tr_df)} pairs")
    bar.update(1)

    if PREP["augment"]:
        aug, n_pos, n_neg = transitive_augment(tr_df, PREP["max_aug_pos"], PREP["max_aug_neg"], CFG["seed"])
        tr_df = pd.concat([tr_df[["question1", "question2", "is_duplicate"]], aug], ignore_index=True)
        tr_df = tr_df.sample(frac=1, random_state=CFG["seed"]).reset_index(drop=True)
        print(f"Augmentation added {n_pos} duplicate + {n_neg} non-duplicate pairs (train only)")
    bar.update(1)

print("Final train/val/test sizes:", len(tr_df), len(va_df), len(te_df), "| train duplicate rate:", round(tr_df.is_duplicate.mean(), 3))

Preprocessing:   0%|          | 0/3 [00:00<?, ?it/s]

Cleaned train: 323478 -> 323468 pairs
Augmentation added 2608 duplicate + 5654 non-duplicate pairs (train only)
Final train/val/test sizes: 331730 40435 40435 | train duplicate rate: 0.368


In [4]:
# ---------------- Tokenizer, dataset, model, optimizer ----------------
with tqdm(total=5, desc="Model setup") as bar:
    tok = AutoTokenizer.from_pretrained(CFG["model_name"]); bar.update(1)

    class PairDS(torch.utils.data.Dataset):
        def __init__(self, d):
            self.q1, self.q2, self.y = d.question1.tolist(), d.question2.tolist(), d.is_duplicate.values
        def __len__(self): return len(self.y)
        def __getitem__(self, i): return self.q1[i], self.q2[i], self.y[i]

    def train_collate(batch):
        q1, q2, y = zip(*batch)
        swap = [random.random() < 0.5 for _ in q1]          # random order swap (augmentation)
        a = [b_ if s else a_ for a_, b_, s in zip(q1, q2, swap)]
        b = [a_ if s else b_ for a_, b_, s in zip(q1, q2, swap)]
        enc = tok(a, b, truncation=True, max_length=CFG["max_len"], padding=True, return_tensors="pt")
        enc["labels"] = torch.tensor(y, dtype=torch.long)
        return enc

    train_dl = DataLoader(PairDS(tr_df), batch_size=CFG["train_bs"], shuffle=True,
                          collate_fn=train_collate, num_workers=2, drop_last=True); bar.update(1)

    model = AutoModelForSequenceClassification.from_pretrained(CFG["model_name"], num_labels=2)
    model = model.float().to(device)     # FP32 master weights (fp16 weights break GradScaler)
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
    unwrap = lambda m: m.module if hasattr(m, "module") else m
    bar.update(1)

    no_decay = ["bias", "LayerNorm.weight"]
    groups = [
        {"params": [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)], "weight_decay": CFG["wd"]},
        {"params": [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)], "weight_decay": 0.0},
    ]
    opt = torch.optim.AdamW(groups, lr=CFG["lr"]); bar.update(1)
    total_steps = len(train_dl) * CFG["epochs"]
    sched = get_linear_schedule_with_warmup(opt, int(CFG["warmup"] * total_steps), total_steps)
    scaler = torch.amp.GradScaler("cuda"); bar.update(1)

print(f"Steps per epoch: {len(train_dl)} | total steps: {total_steps} | {gpu_mem()}")

Model setup:   0%|          | 0/5 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  371MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
classifier.weight                       | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.weight                     | MISSING    | 
pooler.den

Steps per epoch: 5183 | total steps: 15549 | GPU0: 0.7/15 GiB | GPU1: 0.0/15 GiB


In [5]:
# ---------------- Inference + threshold helpers (with progress bars) ----------------
@torch.no_grad()
def predict(q1, q2, tta=True, desc="Predicting"):
    """Return P(duplicate) for lists of questions."""
    model.eval()
    n = len(q1)
    order = np.argsort([len(a) + len(b) for a, b in zip(q1, q2)])        # length-sorted batches = faster
    n_batches = (n + CFG["eval_bs"] - 1) // CFG["eval_bs"]
    passes = 2 if tta else 1
    bar = tqdm(total=n_batches * passes, desc=desc, leave=False)

    def run(A, B):
        res = np.zeros(n, dtype=np.float32)
        for i in range(0, n, CFG["eval_bs"]):
            idx = order[i:i + CFG["eval_bs"]]
            enc = tok([A[j] for j in idx], [B[j] for j in idx], truncation=True,
                      max_length=CFG["max_len"], padding=True, return_tensors="pt").to(device)
            with torch.amp.autocast("cuda"):
                logits = model(**enc).logits
            res[idx] = torch.softmax(logits.float(), -1)[:, 1].cpu().numpy()
            bar.update(1)
        return res

    p = run(q1, q2)
    if tta: p = (p + run(q2, q1)) / 2
    bar.close()
    return p

def best_f1_threshold(y, p):
    best_t, best_f = 0.5, 0.0
    for t in tqdm(np.arange(0.05, 0.95, 0.005), desc="Threshold search", leave=False):
        f = f1_score(y, (p > t).astype(int))
        if f > best_f: best_t, best_f = t, f
    return best_t, best_f

print("Helpers ready")

Helpers ready


In [6]:
# ---------------- Training (3 epochs) ----------------
best_f1, best_thr, best_state = -1, 0.5, None
epoch_bar = tqdm(range(CFG["epochs"]), desc="Epochs", position=0)
for epoch in epoch_bar:
    model.train()
    pbar = tqdm(train_dl, desc=f"Epoch {epoch+1}/{CFG['epochs']} - train", position=1, leave=False, dynamic_ncols=True)
    ema = None
    for step, batch in enumerate(pbar, 1):
        batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
        labels = batch.pop("labels")
        with torch.amp.autocast("cuda"):
            logits = model(**batch).logits
        loss = nn.functional.cross_entropy(logits.float(), labels)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update(); sched.step()
        ema = loss.item() if ema is None else 0.98 * ema + 0.02 * loss.item()
        if step % 20 == 0:
            pbar.set_postfix(loss=f"{ema:.4f}", lr=f"{sched.get_last_lr()[0]:.2e}")

    val_p = predict(va_df.question1.tolist(), va_df.question2.tolist(), tta=CFG["tta"], desc=f"Epoch {epoch+1} - validation")
    thr, f1 = best_f1_threshold(va_df.is_duplicate.values, val_p)
    ll = log_loss(va_df.is_duplicate.values, np.clip(val_p, 1e-6, 1 - 1e-6))
    tqdm.write(f"==== Epoch {epoch+1}: VAL F1 = {f1:.4f} @ threshold {thr:.3f} | logloss = {ll:.4f} | {gpu_mem()}")
    epoch_bar.set_postfix(val_f1=f"{f1:.4f}")

    if f1 > best_f1:
        best_f1, best_thr = f1, thr
        best_state = {k: v.detach().cpu().clone() for k, v in unwrap(model).state_dict().items()}
        torch.save(best_state, "best_model.pt")          # saved every time it improves (safe if session dies)
        tqdm.write("     -> new best, checkpoint saved to best_model.pt")
    gc.collect(); torch.cuda.empty_cache()

unwrap(model).load_state_dict(best_state)
print(f"Best val F1 = {best_f1:.4f} (threshold {best_thr:.3f})")

Epochs:   0%|          | 0/3 [00:00<?, ?it/s]

Epoch 1/3 - train:   0%|          | 0/5183 [00:00<?, ?it/s]

Epoch 1 - validation:   0%|          | 0/316 [00:00<?, ?it/s]

Threshold search:   0%|          | 0/180 [00:00<?, ?it/s]

==== Epoch 1: VAL F1 = 0.8824 @ threshold 0.485 | logloss = 0.2215 | GPU0: 2.8/15 GiB | GPU1: 0.0/15 GiB
     -> new best, checkpoint saved to best_model.pt


...

==== Epoch 2: VAL F1 = 0.8925 @ threshold 0.365 | logloss = 0.2102 | GPU0: 2.8/15 GiB | GPU1: 0.0/15 GiB
     -> new best, checkpoint saved to best_model.pt


Epoch 3/3 - train:   0%|          | 0/5183 [00:00<?, ?it/s]

Epoch 3 - validation:   0%|          | 0/316 [00:00<?, ?it/s]

Threshold search:   0%|          | 0/180 [00:00<?, ?it/s]

==== Epoch 3: VAL F1 = 0.8948 @ threshold 0.450 | logloss = 0.2199 | GPU0: 2.8/15 GiB | GPU1: 0.0/15 GiB
     -> new best, checkpoint saved to best_model.pt
Best val F1 = 0.8948 (threshold 0.450)


In [7]:
# ---------------- Final evaluation on the held-out TEST split ----------------
te_p = predict(te_df.question1.tolist(), te_df.question2.tolist(), tta=CFG["tta"], desc="Test inference")
with tqdm(total=3, desc="Metrics") as bar:
    te_pred = (te_p > best_thr).astype(int); y = te_df.is_duplicate.values; bar.update(1)
    f1_t, acc_t = f1_score(y, te_pred), accuracy_score(y, te_pred); bar.update(1)
    ll_t = log_loss(y, np.clip(te_p, 1e-6, 1 - 1e-6)); bar.update(1)

print(f"TEST F1       : {f1_t:.4f}   (threshold {best_thr:.3f} chosen on validation)")
print(f"TEST accuracy : {acc_t:.4f}")
print(f"TEST logloss  : {ll_t:.4f}\n")
print(classification_report(y, te_pred, target_names=["not duplicate", "duplicate"], digits=4))

out = te_df[["id", "question1", "question2", "is_duplicate"]].copy()
out["pred_prob"], out["pred_label"] = te_p, te_pred
out.to_csv("test_predictions.csv", index=False)
print("Saved test_predictions.csv")

Test inference:   0%|          | 0/316 [00:00<?, ?it/s]

Metrics:   0%|          | 0/3 [00:00<?, ?it/s]

TEST F1       : 0.8912   (threshold 0.450 chosen on validation)
TEST accuracy : 0.9178
TEST logloss  : 0.2302

               precision    recall  f1-score   support

not duplicate     0.9472    0.9210    0.9339     25505
    duplicate     0.8711    0.9123    0.8912     14930

     accuracy                         0.9178     40435
    macro avg     0.9091    0.9166    0.9126     40435
 weighted avg     0.9191    0.9178    0.9181     40435

Saved test_predictions.csv


In [8]:
# ---------------- Stage 2: stacking (DeBERTa probability + structure features) ----------------
# Trains a small LightGBM on the VALIDATION split only (DeBERTa never trained on it, so its probabilities are honest there),
# with 5-fold CV. The TEST split is only used for the final comparison.
import subprocess, sys
try:
    import lightgbm as lgb
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lightgbm"], check=True); import lightgbm as lgb
from collections import defaultdict
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import precision_score, recall_score

STACK = dict(folds=5, seed=42)

def logit(p): p = np.clip(p, 1e-5, 1 - 1e-5); return np.log(p / (1 - p))

# question graph built from ALL pairs in questions.csv (pairs only, NO labels are used)
nbrs = defaultdict(set)
for a, b in zip(df.qid1, df.qid2):
    nbrs[a].add(b); nbrs[b].add(a)

def make_feats(d, p):
    q1 = [s.lower().split() for s in d.question1]; q2 = [s.lower().split() for s in d.question2]
    rows = []
    for a, b, w1, w2 in zip(d.qid1, d.qid2, q1, q2):
        s1, s2 = set(w1), set(w2); common = len(s1 & s2); union = max(len(s1 | s2), 1)
        rows.append((len(w1), len(w2), abs(len(w1) - len(w2)), common, common / union,
                     int(w1[:1] == w2[:1]), int(w1[-1:] == w2[-1:]),                      # text features
                     len(nbrs[a]), len(nbrs[b]), len(nbrs[a] & nbrs[b])))                 # graph features
    F = pd.DataFrame(rows, columns=["w1", "w2", "wdiff", "common", "jaccard", "first_eq", "last_eq",
                                    "deg1", "deg2", "common_nbrs"])
    F["dbert_logit"] = logit(p)
    return F

TEXT = ["dbert_logit", "w1", "w2", "wdiff", "common", "jaccard", "first_eq", "last_eq"]
GRAPH = TEXT + ["deg1", "deg2", "common_nbrs"]

with tqdm(total=3, desc="Stage-2 inputs") as bar:
    val_p = predict(va_df.question1.tolist(), va_df.question2.tolist(), tta=CFG["tta"], desc="Val (best weights)"); bar.update(1)
    Fva, Fte = make_feats(va_df, val_p), make_feats(te_df, te_p); bar.update(1)
    yva, yte = va_df.is_duplicate.values, te_df.is_duplicate.values; bar.update(1)

def report(name, y, p, thr):
    pred = (p > thr).astype(int)
    return dict(model=name, thr=round(float(thr), 3), F1=f1_score(y, pred), acc=accuracy_score(y, pred),
                precision=precision_score(y, pred), recall=recall_score(y, pred),
                logloss=log_loss(y, np.clip(p, 1e-6, 1 - 1e-6)))

def run_stack(cols, name):
    skf = StratifiedKFold(STACK["folds"], shuffle=True, random_state=STACK["seed"])
    oof, te_s = np.zeros(len(Fva)), np.zeros(len(Fte))
    for tr_i, va_i in tqdm(list(skf.split(Fva, yva)), desc=name, leave=False):
        m = lgb.LGBMClassifier(n_estimators=400, learning_rate=0.03, num_leaves=15, min_child_samples=40,
                               subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0, verbose=-1)
        m.fit(Fva.iloc[tr_i][cols], yva[tr_i])
        oof[va_i] = m.predict_proba(Fva.iloc[va_i][cols])[:, 1]
        te_s += m.predict_proba(Fte[cols])[:, 1] / STACK["folds"]
    thr, _ = best_f1_threshold(yva, oof)           # threshold chosen on out-of-fold validation predictions
    return report(name, yte, te_s, thr), te_s

base_thr, _ = best_f1_threshold(yva, val_p)
rows = [report("DeBERTa alone", yte, te_p, base_thr)]
r_text, _ = run_stack(TEXT, "Stack: DeBERTa + text features"); rows.append(r_text)
r_graph, te_graph = run_stack(GRAPH, "Stack: + question-graph features"); rows.append(r_graph)

res = pd.DataFrame(rows).set_index("model")
print("TEST-split comparison (same rows for every line):")
print(res.round(4).to_string())

# precision / recall trade-off for the best stacked model (raise threshold -> higher precision, lower recall)
tab = [report(f"thr={t}", yte, te_graph, t) for t in (0.3, 0.4, 0.5, 0.6, 0.7, 0.8)]
print("\nThreshold trade-off (graph stack):")
print(pd.DataFrame(tab).set_index("model")[["precision", "recall", "F1", "acc"]].round(4).to_string())

Stage-2 inputs:   0%|          | 0/3 [00:00<?, ?it/s]

Val (best weights):   0%|          | 0/316 [00:00<?, ?it/s]

Threshold search:   0%|          | 0/180 [00:00<?, ?it/s]

Stack: DeBERTa + text features:   0%|          | 0/5 [00:00<?, ?it/s]

Threshold search:   0%|          | 0/180 [00:00<?, ?it/s]

Stack: + question-graph features:   0%|          | 0/5 [00:00<?, ?it/s]

Threshold search:   0%|          | 0/180 [00:00<?, ?it/s]

TEST-split comparison (same rows for every line):
                                    thr      F1     acc  precision  recall  logloss
model                                                                              
DeBERTa alone                     0.450  0.8912  0.9178     0.8711  0.9123   0.2302
Stack: DeBERTa + text features    0.455  0.8940  0.9192     0.8672  0.9224   0.1988
Stack: + question-graph features  0.455  0.8941  0.9193     0.8665  0.9236   0.1985

Threshold trade-off (graph stack):
         precision  recall      F1     acc
model                                     
thr=0.3     0.8259  0.9580  0.8870  0.9099
thr=0.4     0.8486  0.9410  0.8924  0.9162
thr=0.5     0.8778  0.9096  0.8934  0.9198
thr=0.6     0.8918  0.8838  0.8878  0.9175
thr=0.7     0.9110  0.8358  0.8718  0.9092
thr=0.8     0.9292  0.7717  0.8432  0.8940


In [9]:
# ---------------- Try your own question pairs ----------------
def is_duplicate(q1, q2):
    p = predict([q1], [q2], tta=True, desc="Predicting")[0]
    return {"probability": round(float(p), 4), "duplicate": bool(p > best_thr)}

for a, b in tqdm([("How can I learn Python quickly?", "What is the fastest way to learn Python?"),
                  ("How can I learn Python quickly?", "What is the capital of France?")], desc="Examples"):
    print(a, "|", b, "->", is_duplicate(a, b))

Examples:   0%|          | 0/2 [00:00<?, ?it/s]

Predicting:   0%|          | 0/2 [00:00<?, ?it/s]

How can I learn Python quickly? | What is the fastest way to learn Python? -> {'probability': 0.599, 'duplicate': True}


Predicting:   0%|          | 0/2 [00:00<?, ?it/s]

How can I learn Python quickly? | What is the capital of France? -> {'probability': 0.7455, 'duplicate': True}
